# Chess commentary student v3 -- Qwen2.5-1.5B LoRA (Kaggle)

Trains a small model to write move commentary from verified engine + board facts, imitating the app's
teacher (gpt-oss-120b with a fact-check pass). Data: `distill/build_v3_split.py` output
(`train.jsonl`, `val.jsonl`, `test.jsonl`), already filtered by the automatic consistency check.

**Before running:** upload the three files as a Kaggle Dataset, attach it, set `DATA_DIR`,
choose **GPU T4 x2**, turn **Internet on**, then Run All (or Save & Run All to run in the background).

In [ ]:
!pip install -q -U peft trl accelerate datasets torchao

In [ ]:
import os
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

import json, random, time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
# ---- Config: edit these ----
STUDENT_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
DATA_DIR = "/kaggle/input/<your-v3-dataset>"      # paste the path from the Input sidebar
OUTPUT_DIR = "/kaggle/working/chess-annotator-lora-v3"

LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

NUM_EPOCHS = 2
LEARNING_RATE = 1e-4
PER_DEVICE_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8            # effective batch 16
MAX_SEQ_LEN = 1024              # v3 prompts are ~650 tokens; the check below fails loudly if any are longer
MAX_TRAIN = None                # e.g. 3000 for a quick trial run
TEST_GEN_BATCH = 16             # batched greedy generation on the test set
MAX_NEW_TOKENS = 260
# ----------------------------
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

## 1. Data

In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

train_records = load_jsonl(f"{DATA_DIR}/train.jsonl")
val_records = load_jsonl(f"{DATA_DIR}/val.jsonl")
test_records = load_jsonl(f"{DATA_DIR}/test.jsonl")
if MAX_TRAIN:
    random.shuffle(train_records)
    train_records = train_records[:MAX_TRAIN]
print(f"train={len(train_records)}  val={len(val_records)}  test={len(test_records)}")
print(train_records[0]["user"][:600], "...\n->", train_records[0]["completion"][:300])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def render_prompt(r):
    msgs = [{"role": "system", "content": r["system"]}, {"role": "user", "content": r["user"]}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def build_dataset(records):
    return Dataset.from_dict({"prompt": [render_prompt(r) for r in records],
                              "completion": [r["completion"] for r in records]})

train_ds = build_dataset(train_records)
val_ds = build_dataset(val_records)

lengths = sorted(len(tokenizer(p + c)["input_ids"]) for p, c in zip(train_ds["prompt"], train_ds["completion"]))
print(f"train sequence length: median {lengths[len(lengths)//2]}, max {lengths[-1]} (MAX_SEQ_LEN={MAX_SEQ_LEN})")
assert lengths[-1] <= MAX_SEQ_LEN, "Some examples exceed MAX_SEQ_LEN -- raise it or they will be truncated"

## 2. Model + LoRA

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)
model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
))
model.print_trainable_parameters()

## 3. Train (validation loss every half epoch)

In [ ]:
steps_per_epoch = max(1, len(train_ds) // (PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS))
eval_every = max(20, steps_per_epoch // 2)

args = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_length=MAX_SEQ_LEN,
    completion_only_loss=True,
    loss_type="nll",                       # trl's default chunked loss misbehaves with PEFT models
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=torch.cuda.is_available(),
    logging_steps=20,
    disable_tqdm=True,                     # print losses as plain text, so background (commit) runs show progress in the log
    eval_strategy="steps", eval_steps=eval_every,
    save_strategy="steps", save_steps=eval_every, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    report_to="none",
    seed=SEED,
)
trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, processing_class=tokenizer)
t0 = time.time()
trainer.train()
print(f"training took {(time.time() - t0) / 60:.0f} min")

In [ ]:
hist = trainer.state.log_history
tr = [(h["step"], h["loss"]) for h in hist if "loss" in h]
ev = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]
plt.figure(figsize=(7, 4))
if tr: plt.plot(*zip(*tr), label="train loss")
if ev: plt.plot(*zip(*ev), marker="o", label="val loss")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title("v3 student: train vs val loss")
plt.savefig(f"{OUTPUT_DIR}/loss_curve.png", dpi=120, bbox_inches="tight"); plt.show()
print("val loss by eval:", [round(x[1], 4) for x in ev])

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

## 4. Test set: generate, then check every annotation against its facts

The checker below is `distill/consistency.py`, embedded verbatim.

In [ ]:
"""Automatic fact-consistency checks for a move annotation.

check(facts, text) compares an annotation against the facts JSON it was
written from (the record built by chess_annotator.app_prompts) and returns a
list of violations -- claims the facts don't support. Used to:
  * filter teacher labels before training (drop annotations with violations),
  * score a student model (share of annotations with zero violations).

The checks are deliberately conservative (few false alarms) rather than
exhaustive: they catch wrong advantage sizes, wrong side, wrong direction,
tactics/structures not in the insights, captures and checks that didn't
happen, and pieces named on squares where they don't stand. Sentences about
hypothetical engine lines ("would", "could", the best move) are skipped for
the direction/capture/check rules.

Pure Python (no python-chess), so it also runs inside the Kaggle notebook.
"""

from __future__ import annotations

import re

PIECES = ("king", "queen", "rook", "bishop", "knight", "pawn")

# phrase regex -> substring that must appear in the insights text for the phrase to be allowed
TACTIC_TERMS = [
    (r"\bpin(s|ned|ning)?\b", ("pin",)),
    (r"\bfork(s|ed|ing)?\b", ("fork",)),
    (r"\bskewer", ("skewer",)),
    (r"\bdiscovered\b", ("discover",)),
    (r"\bdouble check\b", ("double check",)),
    (r"threat(ens|ening)? (of )?mate|mating threat|mate threat", ("mate",)),
    (r"\bsemi-?open\b|\bhalf-?open\b", ("semi-open",)),
    (r"\bopen [a-h]-file\b|\bopen file\b", ("open",)),
    (r"\bhanging pawns?\b", ("hanging pawns",)),
    (r"\bisolated\b|\bisolani\b", ("isolated",)),
    (r"\bpassed pawn|\bpasser\b", ("passed pawn",)),
    (r"\bdoubled pawns?\b", ("doubled pawns",)),
    (r"\bbishop pair\b|\btwo bishops\b", ("bishop pair",)),
    (r"\bopposite[- ]colou?red\b", ("opposite",)),
    (r"\bbad bishop\b", ("'bad' bishop",)),
    (r"\bpawn majority\b", ("majority",)),
    (r"\ben prise\b|\bundefended\b|\bhangs\b|\bleft hanging\b", ("en prise", "not adequately defended", "can itself be captured")),
    (r"\boutpost\b|\bweak square\b|\bhole\b", ()),  # never produced by the detectors
]

WIN_WORDS = re.compile(r"\b(decisive|winning(?! (?:a|the|that|this|his|her|back|an?)\b)|crushing|overwhelming|won position)\b", re.I)
EQUAL_WORDS = re.compile(r"\b(roughly|essentially|about|dead) (equal|level|balanced)\b|\bequal position\b", re.I)
SIDE_BETTER = re.compile(
    r"\b(white|black)(?:['’]s)? (?:is|remains|stands|stays|was) (?:still |now |clearly |much |slightly |comfortably )?"
    r"(?:better|ahead|winning|on top)\b|\badvantage (?:for|to) (white|black)\b", re.I)
WORSEN = re.compile(r"\b(worsen(s|ed|ing)?|costs?|drops?|loses?|slips?|gives? away|squanders?)\b", re.I)
IMPROVE = re.compile(r"\b(improv(es|ed|ing)|gains?|increas(es|ed|ing)|raises?|boosts?|strengthens?)\b", re.I)
HYPOTHETICAL = re.compile(r"\b(would|could|can|cannot|can't|might|instead|should|if|not|without)\b", re.I)
CAPTURE = re.compile(r"\b(captures?|capturing|takes|grabs?|snaps? (off|up)|wins (a|the) (pawn|knight|bishop|rook|queen))\b", re.I)
CHECK = re.compile(r"\b(gives? check|with check|delivers? check|delivering check|checks the king)\b", re.I)
PIECE_ON = re.compile(r"\b(king|queen|rook|bishop|knight|pawn)s? on ([a-h][1-8])\b", re.I)
MATERIAL_WIN = re.compile(r"\b(wins?|winning|gains?|nets?) (a |the )?(pawn|piece|material|exchange)\b|\bmaterial gain\b", re.I)
PLURAL = re.compile(r"\b(both|two|coordinat\w*|connect\w*) (?:the |his |her |white['’]s |black['’]s )?(bishops|knights|rooks)\b", re.I)
FORBIDDEN = re.compile(r"\bcentipawns?\b|\bcp\b", re.I)


def normalize(text: str) -> str:
    """Models often write non-breaking or typographic hyphens/dashes/quotes; normalise them."""
    for ch in "\u2010\u2011\u2012\u2013\u2212":
        text = text.replace(ch, "-")
    return text.replace("\u2019", "'").replace("\u00a0", " ")


def _sentences(text: str) -> list:
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]


def _insights_text(facts: dict) -> str:
    parts = []
    for k in ("move_ideas", "position_context", "best_move_ideas", "opponent_reply_ideas"):
        parts.extend(facts.get(k) or [])
    return " ".join(parts).lower()


def _abs_eval(block: dict):
    if not block:
        return None
    if block.get("mate_in") is not None:
        return 10000
    return abs(block["cp"]) if block.get("cp") is not None else None


def _sign(block: dict):
    if not block:
        return 0
    if block.get("mate_in") is not None:
        return 1 if block["mate_in"] > 0 else -1
    cp = block.get("cp")
    if cp is None or abs(cp) < 50:
        return 0
    return 1 if cp > 0 else -1


def _pieces(facts: dict) -> set:
    out = set()
    for side, desc in (facts.get("pieces_after") or {}).items():
        for part in desc.split(";"):
            if ":" not in part:
                continue
            name, squares = part.split(":", 1)
            name = name.strip().rstrip("s")
            for sq in squares.split(","):
                out.add((name, sq.strip()))
    return out


def check(facts: dict, text: str) -> list:
    """Return a list of human-readable violations (empty list = consistent)."""
    v = []
    if not text or not text.strip():
        return ["empty annotation"]
    text = normalize(text)
    if not text.rstrip().endswith((".", "!", "?", '"', "'", ")")):
        v.append("annotation is cut off mid-sentence")
    low = text.lower()
    ins = _insights_text(facts)

    if FORBIDDEN.search(text):
        v.append("uses 'centipawn'/'cp'")

    # Size of advantage
    before, after = _abs_eval(facts.get("eval_before")), _abs_eval(facts.get("eval_after"))
    if WIN_WORDS.search(text) and not any(x is not None and x > 300 for x in (before, after)):
        v.append(f"calls the position decisive/winning at {facts.get('eval_after', {}).get('pawns')}")
    if EQUAL_WORDS.search(text) and after is not None and after > 100 and (before is None or before > 100):
        v.append(f"calls the position equal at {facts.get('eval_after', {}).get('pawns')}")

    # Which side is better
    signs = {_sign(facts.get("eval_before")), _sign(facts.get("eval_after"))} - {0}
    for m in SIDE_BETTER.finditer(text):
        if re.match(r"\s*(in|on) material", text[m.end():m.end() + 14], re.I):
            continue  # "ahead in material" is about material, not the evaluation
        side = (m.group(1) or m.group(2)).lower()
        claimed = 1 if side == "white" else -1
        if signs and claimed not in signs:
            v.append(f"says {side.title()} is better, but the engine favours the other side")

    # Direction of the move's effect, captures, checks -- only in factual sentences
    change = (facts.get("eval_change_for_mover") or {}).get("cp")
    best = (facts.get("best_move_san") or "").lower()
    is_capture = bool(facts.get("captured_piece")) or "capture" in (facts.get("tags") or [])
    gives_check = "check" in (facts.get("tags") or [])
    for s in _sentences(text):
        sl = s.lower()
        if HYPOTHETICAL.search(s) or (best and best != facts.get("san", "").lower() and best in sl):
            continue
        if change is not None:
            if change >= 40 and WORSEN.search(s) and not IMPROVE.search(s):
                v.append(f"says the move worsened the position, but it gained {change / 100:.2f}")
            if change <= -40 and IMPROVE.search(s) and not WORSEN.search(s) and "evaluation" in sl:
                v.append(f"says the move improved the evaluation, but it lost {-change / 100:.2f}")
        if CAPTURE.search(s) and not is_capture and "exchange" not in ins and "captures" not in ins:
            v.append("describes a capture that did not happen")
        if CHECK.search(s) and not gives_check:
            v.append("describes a check that did not happen")

    # Material won in what is really an exchange
    if "no material is won" in ins and any(MATERIAL_WIN.search(x) and not HYPOTHETICAL.search(x) for x in _sentences(text)):
        v.append("claims material is won, but the capture is an exchange")

    # Plural pieces the mover doesn't have ("coordinate the bishops" with one bishop)
    pieces_desc = (facts.get("pieces_after") or {}).get("White" if facts.get("side") == "white" else "Black", "")
    if pieces_desc:
        for m in PLURAL.finditer(text):
            kind = m.group(2).lower()[:-1]
            n = next((len(part.split(":", 1)[1].split(",")) for part in pieces_desc.split(";")
                      if ":" in part and part.split(":", 1)[0].strip().rstrip("s") == kind), 0)
            if n < 2:
                v.append(f"mentions two {kind}s, but the mover has {n}")

    # Tactical / structural terms must be backed by the insights
    for pattern, needs in TACTIC_TERMS:
        m = re.search(pattern, low)
        if m and not any(n in ins for n in needs):
            v.append(f"mentions '{m.group(0)}' which is not in the insights")

    # Pieces named on squares must exist there (after the move) or be named in the insights
    pieces = _pieces(facts)
    if pieces:
        from_sq = (facts.get("uci") or "")[:2]
        for m in PIECE_ON.finditer(text):
            name, sq = m.group(1).lower(), m.group(2).lower()
            if (name, sq) in pieces or f"{name} on {sq}" in ins or sq == from_sq:
                continue
            v.append(f"names a {name} on {sq}, but there is none")
    return sorted(set(v))

In [ ]:
model.eval()
tokenizer.padding_side = "left"

def generate_batch(records):
    prompts = [render_prompt(r) for r in records]
    enc = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
    return [tokenizer.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip() for o in out]

t0 = time.time()
student = []
for i in range(0, len(test_records), TEST_GEN_BATCH):
    student += generate_batch(test_records[i:i + TEST_GEN_BATCH])
    if (i // TEST_GEN_BATCH) % 10 == 0:
        print(f"  {len(student)}/{len(test_records)}  ({time.time() - t0:.0f}s)")

results = []
for r, s in zip(test_records, student):
    facts = json.loads(r["facts"]); facts["uci"] = r["uci"]
    results.append({**{k: r[k] for k in ("game_id", "source", "ply", "san", "quality_label", "best_move_san")},
                    "teacher": r["completion"], "student": s, "violations": check(facts, s)})
Path(f"{OUTPUT_DIR}/test_outputs.json").write_text(json.dumps(results, indent=1, ensure_ascii=False))

n = len(results)
clean = sum(not x["violations"] for x in results)
print(f"\nStudent consistency: {clean}/{n} ({100 * clean / n:.1f}%) annotations with no violations "
      f"(teacher: 100% by construction -- the data was filtered with the same check)")
from collections import Counter
print(Counter(v.split(",")[0][:55] for x in results for v in x["violations"]).most_common(10))
g = [x for x in results if x["quality_label"] in ("blunder", "mistake") and x["best_move_san"]]
hits = sum(x["best_move_san"] in x["student"] for x in g)
print(f"Blunders/mistakes naming the engine's best move: {hits}/{len(g)} ({100 * hits / max(1, len(g)):.0f}%)")
by_src = {}
for x in results:
    by_src.setdefault("own games" if x["source"] == "shanky8991" else "master games", []).append(not x["violations"])
for k, v in by_src.items():
    print(f"  {k}: {100 * sum(v) / len(v):.1f}% clean ({len(v)} examples)")
for x in random.sample(results, 6):
    print("=" * 70); print(f"{x['san']} [{x['quality_label']}] violations={x['violations']}")
    print("TEACHER:", x["teacher"]); print("STUDENT:", x["student"])

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/chess-annotator-lora-v3", "zip", OUTPUT_DIR)
print("Download: /kaggle/working/chess-annotator-lora-v3.zip (adapter + loss curve + test_outputs.json)")